## Setup

We import bunch of libaries to work with language models and visualise outcomes.

In [ ]:
## This just downloads the pre-trained model, used in collab environment.
!test -d models || \
  (wget -q https://github.com/codingsocialscience/marxist-llm-tutorial/releases/latest/download/model.zip && \
   unzip -q -o model.zip)

In [ ]:
%pip install transformers torch ipywidgets pandas

In [ ]:
import transformers

from transformers import pipeline
import torch

In [ ]:
transformers.enable_full_determinism( 0 )

## Set up models

Our baseline model is the model used for training (so that we compare apples to apples),
_gpt2_ is not the best model out there but for a demonstration sufficiently good.
`finetuned` model loads the spesifically trained model.

In [ ]:
model_name = "openai-community/gpt2"

In [ ]:
finetuned = pipeline('text-generation', model = f"./models/{model_name.replace('/', '_')}-finetuned-causal-model/")
baseline = pipeline('text-generation', model = model_name )

## Working with the models

We give different prompts to the model and print out the outcomes.

In [ ]:
prompt = "The purpose of companies is to"
print( finetuned(prompt)[0]['generated_text'] )
print( baseline(prompt)[0]['generated_text'] )

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd

text_input = widgets.Textarea(
    placeholder='Enter prompt here...',
    layout=widgets.Layout(width='100%', height='80px')
)
run_button = widgets.Button(description='Run', button_style='primary')
output_area = widgets.Output()

def on_run(b):
    prompt = text_input.value.strip() + ' '
    if not prompt:
        return
    with output_area:
        clear_output()
        print("Running... please wait")

        rows = []
    for i in range(5):
        ft_result = finetuned(prompt)
        bl_result = baseline(prompt)
        rows.append({
            'Run': i + 1,
            'Finetuned': ft_result[0]['generated_text'],
            'Baseline': bl_result[0]['generated_text'],
        })

    df = pd.DataFrame(rows).set_index('Run')

    with output_area:
        clear_output()
        with pd.option_context('display.max_colwidth', None):
            display(df)

run_button.on_click(on_run)
display(widgets.VBox([text_input, run_button, output_area]))
